###测 试

In [ ]:
# Load the Model 
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "/root/autodl-tmp/meta_v2/RM-R1-Dpsk-Distilled-7B_v2-LR1.0e-6/global_step_93/actor"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto" # or specify the specific device map if needed
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards: 100%|██████████| 7/7 [00:27<00:00,  3.96s/it]


In [ ]:
#"winner": "model_a",
# Single Turn Example - from Reward Bench 
prompt = "我昨天下单的空调预约今天14:00-16:00安装，师傅爽约还不接电话，我请了半天假白等了。这样的服务谁负责？需要我重新预约吗？"
answer_a = "真抱歉让您白等。我这边马上联系安装商催派，并为您改约今晚或明早的优先时段，确认后第一时间回电；额外补偿30元券。请留方便接听的号码和时段。若再次爽约可申请延误赔付。" # Accepted
answer_b = "非常抱歉给您带来不便，您的反馈我们已记录并会持续优化服务。安装以系统排期为准，请您耐心等待师傅联系，届时会以短信或电话通知。给您造成困扰再次抱歉，感谢理解与支持。如需改约可在订单页自助操作。"  # Rejected

user_prompt_single = REASONING_SINGLE_PROMPT_TEMPLATE.format(
    question=prompt,
    answer_a=answer_a,
    answer_b=answer_b
) 

conversation = [
    {"role":"user", "content": user_prompt_single}
]

input_ids = tokenizer.apply_chat_template(
    conversation, 
    tokenize=True, 
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

generation = model.generate(
    input_ids=input_ids,
    max_new_tokens=8192, # For optimal benchmarking, set this to unlimited (e.g., 50000)
    do_sample=False,
)

completion = tokenizer.decode(
    generation[0][len(input_ids[0]):], 
    skip_special_tokens=True, 
    clean_up_tokenization_spaces=True
)

print(completion)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Okay, so I need to evaluate which of the two chatbots, A or B, provided a better response to the client's question about how to detail a car. Let me start by reading both responses carefully.

First, Chatbot A's response is quite detailed. It breaks down the process into exterior and interior detailing, each with several steps. The exterior part includes washing, drying, clay bar treatment, polishing, waxing, and cleaning windows. The interior steps cover removing trash, vacuuming, shampooing, cleaning hard surfaces, windows again, air vents, and final touches. There are also additional tips provided, which is helpful. This seems comprehensive and thorough, covering almost every aspect of car detailing.

On the other hand, Chatbot B's response is much shorter. It mentions washing the exterior and interior, polishing and waxing the exterior, and some interior steps like vacuuming, cleaning upholstery, polishing the dashboard, and dusting. It also notes that exterior polishing and waxing